In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
import pickle

from sklearn.cluster import SpectralClustering, KMeans

from torchvision.datasets import MNIST

from dataset_builder import *
from clustering_utils import *
from Siamese_networks import *

In [2]:
from sklearn.metrics.cluster import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances

In [3]:
class SiameseNetwork(nn.Module):
    def __init__(self, embedding_size = 2):
        super(SiameseNetwork_Couple, self).__init__()

        # CNN Layers
        self.cnn1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, stride=2),

            nn.Conv2d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, stride=2)
        )

        # fully connected layers
        self.fc1 = nn.Sequential(
            
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(inplace=True),

            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),

            nn.Linear(512, embedding_size)
        )

    def forward_once(self, x):
        output = self.cnn1(x)
        output = output.view(output.size()[0], -1)
        output = self.fc1(output)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

class ContrastiveWeightedLoss(torch.nn.Module):
    def __init__(self, margin=2.0, w = 1): # prova ad aumentare il margin
        super(ContrastiveWeightedLoss, self).__init__()
        self.margin = margin
        self.w = w
    def forward(self, output1, output2, label):
      euclidean_distance = F.pairwise_distance(output1, output2, keepdim = True)

      loss_contrastive = torch.mean((label) * 0.5 * torch.pow(euclidean_distance, 2) +
                                    (1-label) * 0.5 * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))*self.w

      return loss_contrastive

In [4]:
Triplet_SSN_1 = pickle.load(open(r'models\SN_data1%_Triplets.pkl','rb'))
Triplet_SSN_5 = pickle.load(open(r'models\SN_data5%_Triplets.pkl','rb'))
Triplet_SSN_10 = pickle.load(open(r'models\SN_data10%_Triplets.pkl','rb'))
Triplet_SSN_20 = pickle.load(open(r'models\SN_data20%_Triplets.pkl','rb'))

Couple_SSN_1 = pickle.load(open(r'models/SSN (56acc-128bs).pkl','rb'))

Fm_Triplet_SSN_20 = pickle.load(open(r'models\F_MNIST_models\SN_data20%_Triplets_Fmnist.pkl','rb'))

In [5]:
test_loader = FMNIST_data_loader()

In [11]:
def recall_at_k(embeddings, labels, ks=[1, 2, 4, 8]):
    
    dists = pairwise_distances(embeddings, metric='euclidean')
    
    # Disattiva la diagonale (self-matches)
    np.fill_diagonal(dists, np.inf)
    
    recalls = {}
    for k in ks:
        nearest_indices = np.argsort(dists, axis=1)[:, :k]

        # Confronta: almeno 1 vicino ha la stessa etichetta?
        correct = 0
        for i in range(len(labels)):
            neighbors_labels = labels[nearest_indices[i]]
            if labels[i] in neighbors_labels:
                correct += 1
                
        recall = correct / len(labels)
        recalls[f'R@{k}'] = recall

    return recalls

In [12]:
def evaluate_models(list_of_models, test_loader, true_n_clusters):
    for model in list_of_models:
        embedding_set, labels = create_embedding_dataset(model, test_loader)
        
        clustering = SpectralClustering(n_clusters=true_n_clusters, affinity='nearest_neighbors', assign_labels='kmeans', random_state=0)

        cluster_labels = clustering.fit_predict(embedding_set)
        nmi = normalized_mutual_info_score(labels, cluster_labels)
        ari = adjusted_rand_score(labels, cluster_labels)
        sil_score = silhouette_score(embedding_set, cluster_labels)

        recalls = recall_at_k(embedding_set, labels)

        print(print(type(model)))
        print(f"NMI for model : {nmi}")
        print(f"ARI for model : {ari}")
        print(f"Silouhette score for model : {sil_score}\n")
        print(f"recalls:")
        print(recalls)
        print("-------------------------------------------")


In [13]:
evaluate_models([Fm_Triplet_SSN_20], test_loader, 10)

<class 'Siamese_networks.SiameseNetwork_Triplet'>
None
NMI for model : 0.6928615927116708
ARI for model : 0.5645139391130932
Silouhette score for model : 0.37426039576530457

recalls:
{'R@1': 0.769, 'R@2': 0.8573, 'R@4': 0.9155, 'R@8': 0.9512}
-------------------------------------------
